In [2]:
!pip install requests pandas plotly


In [ ]:
import requests, pandas as pd
import os

APP_ID  = os.environ.get('ADZUNA_APP_ID')
APP_KEY = os.environ.get('ADZUNA_APP_KEY')

url = 'https://api.adzuna.com/v1/api/jobs/us/search/1'
params = {
    'app_id': APP_ID,
    'app_key': APP_KEY,
    'results_per_page': 50,
    'what': 'data analyst',
    'where': 'San Francisco'
}

r = requests.get(url, params=params)
print(f"HTTP Status: {r.status_code}")
data = r.json()
print(f"Got {len(data.get('results', []))} jobs")
print(data['results'][0]['title'])

In [17]:
import os

APP_ID  = os.environ.get('ADZUNA_APP_ID',  'YOUR_APP_ID')
APP_KEY = os.environ.get('ADZUNA_APP_KEY', 'YOUR_APP_KEY')

In [4]:
jobs = [
    {
        'title':       j.get('title', ''),
        'company':     j.get('company', {}).get('display_name', ''),
        'description': j.get('description', ''),
        'salary_min':  j.get('salary_min', 0),
        'salary_max':  j.get('salary_max', 0)
    }
    for j in data['results']
]

df = pd.DataFrame(jobs)
print(df.shape)
df.head(3)

(50, 5)


,title,company,description,salary_min,salary_max
0,Business Data Analyst,TEKsystems c/o Allegis Group,Description We are seeking a Product / Data An...,128456.64,128456.64
1,Data Analyst,System One,Job Title : Data Analyst Location: San Francis...,104566.05,104566.05
2,Business Data Analyst,TEKsystems,Description We are seeking a Product / Data An...,140202.42,140202.42


In [10]:
import time

all_jobs = []
page = 1
MAX_JOBS_TO_PULL = 500

while True:
    if len(all_jobs) >= MAX_JOBS_TO_PULL:
        print(f"Target of {MAX_JOBS_TO_PULL} jobs reached. Stopping.")
        break

    # Construct the URL with the current page number
    current_url = f'https://api.adzuna.com/v1/api/jobs/us/search/{page}'

    r = requests.get(
        current_url,
        params=params # Use the existing params dictionary
    )
    data = r.json()
    current_page_results = data.get('results', [])

    if not current_page_results:
        print(f"Page {page}: No more results found from the API. Stopping.")
        break

    # Add only enough jobs to reach the MAX_JOBS_TO_PULL
    for job in current_page_results:
        if len(all_jobs) < MAX_JOBS_TO_PULL:
            all_jobs.append(job)
        else:
            break # Stop adding if max jobs is reached mid-page

    print(f'Page {page}: {len(all_jobs)} total jobs collected so far', end='\r')

    page += 1
    time.sleep(0.5)

print(f'\nDone: {len(all_jobs)} jobs pulled (Target: {MAX_JOBS_TO_PULL})')

Page 6: No more results found from the API. Stopping.

Done: 246 jobs pulled (Target: 500)


In [11]:
df_all = pd.DataFrame([
    {
        'title':       j.get('title', ''),
        'company':     j.get('company', {}).get('display_name', ''),
        'description': j.get('description', ''),
        'salary_min':  j.get('salary_min', 0),
        'salary_max':  j.get('salary_max', 0)
    }
    for j in all_jobs
])

df_all.to_csv('jobs_raw.csv', index=False)
print(f'Saved {len(df_all)} rows')

Saved 246 rows


In [12]:
from collections import Counter

skills = [
    'sql', 'python', 'tableau', 'power bi', 'excel',
    'looker', 'snowflake', 'bigquery', 'dbt', 'aws',
    'machine learning', 'pandas', 'google analytics',
    'chatgpt', 'copilot', 'openai', 'ai tools',
    'prompt engineering', 'llm', 'gpt'
]

counts = Counter()
for desc in df_all['description']:
    dl = desc.lower()
    for s in skills:
        if s in dl:
            counts[s] += 1

for s, n in counts.most_common():
    print(f'{s:25} {n:4}  ({round(n/len(df_all)*100)}%)')

excel                       15  (6%)
machine learning             8  (3%)
sql                          5  (2%)
tableau                      4  (2%)
llm                          3  (1%)
chatgpt                      2  (1%)
gpt                          2  (1%)
openai                       2  (1%)
python                       1  (0%)
aws                          1  (0%)
ai tools                     1  (0%)


In [13]:
import plotly.express as px

top = counts.most_common(15)
df_s = pd.DataFrame(top, columns=['skill', 'count'])
df_s['pct'] = (df_s['count'] / len(df_all) * 100).round(1)
df_s = df_s.sort_values('pct')

fig = px.bar(
    df_s,
    x='pct',
    y='skill',
    orientation='h',
    title=f'Top Skills in {len(df_all)} SF Data Analyst Postings',
    labels={'pct': '% of job postings', 'skill': ''}
)
fig.show()
fig.write_html('skills_chart.html')

In [14]:
df_sal = df_all[
    (df_all.salary_min > 20000) & (df_all.salary_max > 20000)
].copy()

df_sal['mid'] = (df_sal.salary_min + df_sal.salary_max) / 2

print(f'Jobs with salary listed: {len(df_sal)} ({round(len(df_sal)/len(df_all)*100)}%)')
print(f'Median salary: ${df_sal["mid"].median():,.0f}')
print(f'Range: ${df_sal["mid"].min():,.0f} – ${df_sal["mid"].max():,.0f}')

Jobs with salary listed: 244 (99%)
Median salary: $144,500
Range: $53,485 – $356,500
